# Atlanta to Birmingham: two-branch GPU TCN transfer model

This notebook is dedicated exclusively to Atlanta and Birmingham. It builds a city-specific baseline MLP for each city, trains a weather-residual TCN only on Atlanta, and evaluates the transfer on Birmingham during 2023.

The climate-similarity analysis supports using temperature, precipitation, dew point, and relative humidity as the core shared weather inputs. Pressure and visibility are deliberately excluded because their cross-city similarity is poor.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
from pathlib import Path
import json, platform, sys
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

RAW_DATA_DIR = Path("../LSTM_GPU_TRAINING/LSTM_DATA_PREP/data")
NOTEBOOK_DIR = Path.cwd()
RESULT_DIR = NOTEBOOK_DIR / 'atlanta_birmingham_two_branch_results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CITY = 'atlanta'
TARGET_CITY = 'birmingham'
TRAIN_END = pd.Timestamp('2021-12-31 23:00:00')
VALIDATION_END = pd.Timestamp('2022-12-31 23:00:00')
LOOKBACK_HOURS = 168
BATCH_SIZE = 64
BASELINE_EPOCHS = 80
TCN_EPOCHS = 100
HALF_LIFE_YEARS = 2.0
SEED = 42
USE_MIXED_PRECISION = True

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
assert RAW_DATA_DIR.exists(), f'Raw CSV folder not found: {RAW_DATA_DIR}'
print('Results:', RESULT_DIR)


Results: D:\Downloads\BTP\model_training\lstm_training_2\neural_model\atlanta_birmingham_two_branch_results


## GPU and library sanity check

This must pass before training. If TensorFlow cannot execute a real operation on GPU:0, the notebook stops. CPU is still used for file reading and dataset bookkeeping; neural network computations are placed on the GPU.


In [2]:
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('joblib:', joblib.__version__)
print('TensorFlow:', tf.__version__)
print('Keras:', tf.keras.__version__)
print('Built with CUDA:', tf.test.is_built_with_cuda())

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No TensorFlow GPU was detected. Training is stopped; activate the CUDA-enabled environment and restart the kernel.')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print('Memory growth note:', exc)
tf.config.set_soft_device_placement(False)
print('Physical GPUs:', gpus)
for gpu in gpus:
    print('GPU details:', tf.config.experimental.get_device_details(gpu))
with tf.device('/GPU:0'):
    probe = tf.linalg.matmul(tf.ones((512, 512)), tf.ones((512, 512)))
assert 'GPU' in probe.device.upper(), f'GPU probe ran on {probe.device}'
print('GPU execution verified:', probe.device)
del probe
if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Precision policy:', tf.keras.mixed_precision.global_policy())


Python: 3.10.20
Platform: Windows-10-10.0.26200-SP0
NumPy: 1.26.4
pandas: 2.3.3
scikit-learn: 1.7.2
joblib: 1.5.3
TensorFlow: 2.10.0
Keras: 2.10.0
Built with CUDA: True
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU details: {'device_name': 'NVIDIA GeForce RTX 3050 6GB Laptop GPU', 'compute_capability': (8, 6)}
GPU execution verified: /job:localhost/replica:0/task:0/device:GPU:0
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU, compute capability 8.6
Precision policy: <Policy "mixed_float16">


In [3]:
# Shared weather-variable selection based on the Atlanta-Birmingham similarity analysis.
# Similarity scores: temperature 98.4, precipitation 94.7, dew point 93.2, relative humidity 85.1.
CORE_WEATHER = ['temperature_c', 'dew_point_c', 'relative_humidity_percent', 'hourly_precipitation_cm']
VALID_RANGES = {
    'temperature_c': (-60, 60),
    'dew_point_c': (-80, 50),
    'relative_humidity_percent': (0, 100),
    'hourly_precipitation_cm': (0, 100),
}

def load_master_city(city):
    path = RAW_DATA_DIR / f'{city}_master.csv'
    frame = pd.read_csv(path, parse_dates=['timestamp'])
    required = {'timestamp', 'hourly_load_mw', *CORE_WEATHER}
    assert required <= set(frame.columns), f'{path.name} missing columns: {required - set(frame.columns)}'
    frame = frame.sort_values('timestamp').drop_duplicates('timestamp').set_index('timestamp')
    frame = frame[['hourly_load_mw'] + CORE_WEATHER].asfreq('h')
    frame['hourly_load_mw'] = pd.to_numeric(frame['hourly_load_mw'], errors='coerce').where(lambda x: x > 0)
    for column in CORE_WEATHER:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
        low, high = VALID_RANGES[column]
        frame.loc[~frame[column].between(low, high), column] = np.nan
    # Weather uses previous observations only; load targets are never imputed.
    frame[CORE_WEATHER] = frame[CORE_WEATHER].ffill()
    frame = frame.reset_index().rename(columns={'hourly_load_mw': 'load'}).sort_values('timestamp').reset_index(drop=True)
    time = frame['timestamp']
    frame['hour_sin'] = np.sin(2*np.pi*time.dt.hour/24)
    frame['hour_cos'] = np.cos(2*np.pi*time.dt.hour/24)
    frame['dow_sin'] = np.sin(2*np.pi*time.dt.dayofweek/7)
    frame['dow_cos'] = np.cos(2*np.pi*time.dt.dayofweek/7)
    frame['month_sin'] = np.sin(2*np.pi*(time.dt.month-1)/12)
    frame['month_cos'] = np.cos(2*np.pi*(time.dt.month-1)/12)
    frame['is_weekend'] = (time.dt.dayofweek >= 5).astype('float32')
    frame['years_since_start'] = (time-time.min()).dt.total_seconds()/(365.25*24*3600)
    frame['cooling_degree_c'] = np.clip(frame['temperature_c']-18.0, 0, None)
    frame['heating_degree_c'] = np.clip(18.0-frame['temperature_c'], 0, None)
    frame['temp_x_humidity'] = frame['temperature_c']*frame['relative_humidity_percent']
    weather_features = CORE_WEATHER + ['cooling_degree_c', 'heating_degree_c', 'temp_x_humidity']
    assert frame['timestamp'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
    return frame, weather_features

atlanta, WEATHER_FEATURES = load_master_city(SOURCE_CITY)
birmingham, birmingham_weather_features = load_master_city(TARGET_CITY)
assert WEATHER_FEATURES == birmingham_weather_features
print('Atlanta rows:', len(atlanta), 'missing load hours:', atlanta.load.isna().sum())
print('Birmingham rows:', len(birmingham), 'missing load hours:', birmingham.load.isna().sum())
print('Transferred weather features:', WEATHER_FEATURES)


Atlanta rows: 70128 missing load hours: 48
Birmingham rows: 70128 missing load hours: 48
Transferred weather features: ['temperature_c', 'dew_point_c', 'relative_humidity_percent', 'hourly_precipitation_cm', 'cooling_degree_c', 'heating_degree_c', 'temp_x_humidity']


In [4]:
# Layer 1: one structural baseline MLP per city. It does not see weather variables.
BASELINE_FEATURES = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'years_since_start']

def split_masks(frame):
    train = frame.timestamp <= TRAIN_END
    validation = (frame.timestamp > TRAIN_END) & (frame.timestamp <= VALIDATION_END)
    test = frame.timestamp > VALIDATION_END
    return train, validation, test

def time_weights(timestamps):
    age_years = (TRAIN_END-timestamps).dt.total_seconds().clip(lower=0)/(365.25*24*3600)
    return np.exp(-np.log(2)*age_years/HALF_LIFE_YEARS).astype('float32')

def baseline_mlp():
    inputs = tf.keras.Input((len(BASELINE_FEATURES),), name='calendar_trend_inputs')
    x = tf.keras.layers.Dense(64, activation='relu')(inputs)
    x = tf.keras.layers.Dropout(0.10)(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    outputs = tf.keras.layers.Dense(1, dtype='float32', name='scaled_log_baseline')(x)
    return tf.keras.Model(inputs, outputs, name='city_structural_baseline')

def fit_city_baseline(frame, city):
    train, validation, _ = split_masks(frame)
    usable = frame[BASELINE_FEATURES + ['load']].notna().all(axis=1)
    train, validation = train & usable, validation & usable
    x_scaler, y_scaler = StandardScaler(), StandardScaler()
    X_train = x_scaler.fit_transform(frame.loc[train, BASELINE_FEATURES])
    X_validation = x_scaler.transform(frame.loc[validation, BASELINE_FEATURES])
    y_train = y_scaler.fit_transform(np.log(frame.loc[train, ['load']])).ravel()
    y_validation = y_scaler.transform(np.log(frame.loc[validation, ['load']])).ravel()
    with tf.device('/GPU:0'):
        model = baseline_mlp()
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    model.fit(X_train, y_train, sample_weight=time_weights(frame.loc[train, 'timestamp']),
              validation_data=(X_validation, y_validation), epochs=BASELINE_EPOCHS, batch_size=256,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)
    all_valid = frame[BASELINE_FEATURES].notna().all(axis=1)
    log_baseline = np.full(len(frame), np.nan)
    log_baseline[all_valid] = y_scaler.inverse_transform(model.predict(x_scaler.transform(frame.loc[all_valid, BASELINE_FEATURES]), verbose=0)).ravel()
    output = frame.copy()
    output['log_baseline'] = log_baseline
    output['log_residual'] = np.log(output['load']) - output['log_baseline']
    city_dir = RESULT_DIR / city
    city_dir.mkdir(exist_ok=True)
    model.save(city_dir / 'baseline_mlp.keras')
    joblib.dump(x_scaler, city_dir / 'baseline_feature_scaler.joblib')
    joblib.dump(y_scaler, city_dir / 'baseline_target_scaler.joblib')
    return output

atlanta = fit_city_baseline(atlanta, SOURCE_CITY)
birmingham = fit_city_baseline(birmingham, TARGET_CITY)
print('Baseline parameter count:', tf.keras.models.load_model(RESULT_DIR / SOURCE_CITY / 'baseline_mlp.keras', compile=False).count_params())


Baseline parameter count: 2689


In [5]:
# Layer 2 inputs: shared weather plus causal, baseline-relative load history.
RESIDUAL_FEATURES = ['residual_lag_0h', 'residual_lag_1h', 'residual_lag_24h', 'residual_lag_168h', 'residual_mean_24h', 'residual_std_24h']

def add_residual_history(frame):
    output = frame.copy()
    r = output['log_residual']
    output['residual_lag_0h'] = r
    output['residual_lag_1h'] = r.shift(1)
    output['residual_lag_24h'] = r.shift(24)
    output['residual_lag_168h'] = r.shift(168)
    output['residual_mean_24h'] = r.shift(1).rolling(24, min_periods=24).mean()
    output['residual_std_24h'] = r.shift(1).rolling(24, min_periods=24).std()
    return output

atlanta = add_residual_history(atlanta)
birmingham = add_residual_history(birmingham)
TCN_FEATURES = WEATHER_FEATURES + RESIDUAL_FEATURES
train_mask, _, _ = split_masks(atlanta)
scaler_rows = train_mask & atlanta[TCN_FEATURES + ['log_residual']].notna().all(axis=1)
feature_scaler = StandardScaler().fit(atlanta.loc[scaler_rows, TCN_FEATURES])
residual_scaler = StandardScaler().fit(atlanta.loc[scaler_rows, ['log_residual']])

def scale_for_tcn(frame):
    X = feature_scaler.transform(frame[TCN_FEATURES]).astype('float32')
    y = residual_scaler.transform(frame[['log_residual']]).ravel().astype('float32')
    return X, y

X_atlanta, y_atlanta = scale_for_tcn(atlanta)
X_birmingham, y_birmingham = scale_for_tcn(birmingham)
print('TCN feature count:', len(TCN_FEATURES), '| features:', TCN_FEATURES)


TCN feature count: 13 | features: ['temperature_c', 'dew_point_c', 'relative_humidity_percent', 'hourly_precipitation_cm', 'cooling_degree_c', 'heating_degree_c', 'temp_x_humidity', 'residual_lag_0h', 'residual_lag_1h', 'residual_lag_24h', 'residual_lag_168h', 'residual_mean_24h', 'residual_std_24h']


In [6]:
# Create memory-efficient sequence datasets. Target p uses only input hours p-168 through p-1.
def positions_for_split(frame, X, y, split):
    train, validation, test = split_masks(frame)
    selected = {'train': train, 'validation': validation, 'test': test}[split].to_numpy()
    finite = np.isfinite(X).all(axis=1) & np.isfinite(y)
    cumulative_bad = np.concatenate(([0], np.cumsum(~finite)))
    return np.asarray([p for p in np.flatnonzero(selected)
                       if p >= LOOKBACK_HOURS and cumulative_bad[p+1]-cumulative_bad[p-LOOKBACK_HOURS] == 0], dtype=int)

def sequence_dataset(X, y, positions, training=False, sample_weight=None):
    def batches():
        ids = positions.copy()
        if training:
            np.random.default_rng(SEED).shuffle(ids)
        for start in range(0, len(ids), BATCH_SIZE):
            p = ids[start:start+BATCH_SIZE]
            xb = np.stack([X[i-LOOKBACK_HOURS:i] for i in p]).astype('float32')
            yb = y[p].astype('float32')
            if sample_weight is None:
                yield xb, yb
            else:
                yield xb, yb, sample_weight[p].astype('float32')
    x_spec = tf.TensorSpec((None, LOOKBACK_HOURS, len(TCN_FEATURES)), tf.float32)
    y_spec = tf.TensorSpec((None,), tf.float32)
    signature = (x_spec, y_spec) if sample_weight is None else (x_spec, y_spec, tf.TensorSpec((None,), tf.float32))
    return tf.data.Dataset.from_generator(batches, output_signature=signature).prefetch(tf.data.AUTOTUNE)

atlanta_train_positions = positions_for_split(atlanta, X_atlanta, y_atlanta, 'train')
atlanta_validation_positions = positions_for_split(atlanta, X_atlanta, y_atlanta, 'validation')
birmingham_test_positions = positions_for_split(birmingham, X_birmingham, y_birmingham, 'test')
train_dataset = sequence_dataset(X_atlanta, y_atlanta, atlanta_train_positions, training=True, sample_weight=time_weights(atlanta.timestamp).to_numpy())
validation_dataset = sequence_dataset(X_atlanta, y_atlanta, atlanta_validation_positions)
test_dataset = sequence_dataset(X_birmingham, y_birmingham, birmingham_test_positions)
print('Atlanta training sequences:', len(atlanta_train_positions))
print('Atlanta validation sequences:', len(atlanta_validation_positions))
print('Birmingham test sequences:', len(birmingham_test_positions))


Atlanta training sequences: 51552
Atlanta validation sequences: 8760
Birmingham test sequences: 8760


In [7]:
# The Atlanta-trained weather-residual TCN. Causal dilated convolutions prevent future inputs.
def tcn_block(x, filters, dilation):
    shortcut = x
    for _ in range(2):
        x = tf.keras.layers.Conv1D(filters, 3, padding='causal', dilation_rate=dilation, activation='relu')(x)
        x = tf.keras.layers.SpatialDropout1D(0.10)(x)
    if shortcut.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv1D(filters, 1, padding='same')(shortcut)
    return tf.keras.layers.Activation('relu')(tf.keras.layers.Add()([shortcut, x]))

def weather_residual_tcn():
    inputs = tf.keras.Input((LOOKBACK_HOURS, len(TCN_FEATURES)), name='atlanta_weather_residual_history')
    x = inputs
    for dilation in [1, 2, 4, 8, 16, 32, 64]:
        x = tcn_block(x, 64, dilation)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.20)(x)
    outputs = tf.keras.layers.Dense(1, dtype='float32', name='scaled_next_hour_residual')(x)
    return tf.keras.Model(inputs, outputs, name='atlanta_weather_residual_tcn')

with tf.device('/GPU:0'):
    tcn = weather_residual_tcn()
    tcn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    model_probe = tcn(tf.zeros((1, LOOKBACK_HOURS, len(TCN_FEATURES))))
assert 'GPU' in model_probe.device.upper(), f'TCN probe ran on {model_probe.device}'
print('TCN GPU execution verified:', model_probe.device)
print('TCN parameters:', tcn.count_params())
tcn.summary()


TCN GPU execution verified: /job:localhost/replica:0/task:0/device:GPU:0
TCN parameters: 168257
Model: "atlanta_weather_residual_tcn"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 atlanta_weather_residual_histo  [(None, 168, 13)]   0           []                               
 ry (InputLayer)                                                                                  
                                                                                                  
 conv1d (Conv1D)                (None, 168, 64)      2560        ['atlanta_weather_residual_histor
                                                                 y[0][0]']                        
                                                                                                  
 spatial_dropout1d (SpatialDrop  (None, 168, 64)     0           ['conv1d[

In [8]:
# Train only on Atlanta. Keras data-pipeline bookkeeping remains on CPU; GPU-resident model layers train on GPU.
transfer_dir = RESULT_DIR / 'atlanta_to_birmingham'
transfer_dir.mkdir(exist_ok=True)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(transfer_dir / 'best_atlanta_weather_tcn.keras', monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    tf.keras.callbacks.CSVLogger(transfer_dir / 'atlanta_weather_tcn_history.csv'),
]
history = tcn.fit(train_dataset, validation_data=validation_dataset, epochs=TCN_EPOCHS, callbacks=callbacks, verbose=1)
tcn.save(transfer_dir / 'final_atlanta_weather_tcn.keras')
joblib.dump(feature_scaler, transfer_dir / 'atlanta_feature_scaler.joblib')
joblib.dump(residual_scaler, transfer_dir / 'atlanta_residual_scaler.joblib')
config = {'source_city': SOURCE_CITY, 'target_city': TARGET_CITY, 'train_end': str(TRAIN_END), 'validation_end': str(VALIDATION_END), 'test_period': '2023', 'lookback_hours': LOOKBACK_HOURS, 'time_weight_half_life_years': HALF_LIFE_YEARS, 'weather_features': WEATHER_FEATURES, 'residual_features': RESIDUAL_FEATURES}
(transfer_dir / 'experiment_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')


Epoch 1/100
806/806 [==============================] - 34s 35ms/step - loss: 0.1940 - mae: 0.5077 - val_loss: 0.2160 - val_mae: 0.3518 - lr: 0.0010
Epoch 2/100
806/806 [==============================] - 26s 32ms/step - loss: 0.0599 - mae: 0.2876 - val_loss: 0.1033 - val_mae: 0.2362 - lr: 0.0010
Epoch 3/100
806/806 [==============================] - 26s 32ms/step - loss: 0.0395 - mae: 0.2347 - val_loss: 0.0656 - val_mae: 0.1813 - lr: 0.0010
Epoch 4/100
806/806 [==============================] - 26s 32ms/step - loss: 0.0323 - mae: 0.2113 - val_loss: 0.0610 - val_mae: 0.1809 - lr: 0.0010
Epoch 5/100
806/806 [==============================] - 24s 30ms/step - loss: 0.0289 - mae: 0.1995 - val_loss: 0.0560 - val_mae: 0.1693 - lr: 0.0010
Epoch 6/100
806/806 [==============================] - 25s 31ms/step - loss: 0.0260 - mae: 0.1898 - val_loss: 0.0546 - val_mae: 0.1660 - lr: 0.0010
Epoch 7/100
806/806 [==============================] - 26s 32ms/step - loss: 0.0247 - mae: 0.1830 - val_loss: 0.

606

In [9]:
# Final transfer evaluation on unseen Birmingham 2023 demand.
best_tcn = tf.keras.models.load_model(transfer_dir / 'best_atlanta_weather_tcn.keras')
predicted_scaled_residual = best_tcn.predict(test_dataset, verbose=1).ravel()
predicted_residual = residual_scaler.inverse_transform(predicted_scaled_residual.reshape(-1, 1)).ravel()
p = birmingham_test_positions
predicted_log_load = birmingham.log_baseline.to_numpy()[p] + predicted_residual
predicted_mw = np.exp(predicted_log_load)
actual_mw = birmingham.load.to_numpy()[p]
metrics = {
    'experiment': 'Atlanta weather-residual TCN to Birmingham 2023',
    'MAE_MW': float(np.mean(np.abs(actual_mw-predicted_mw))),
    'RMSE_MW': float(np.sqrt(np.mean((actual_mw-predicted_mw)**2))),
    'MAPE_percent': float(np.mean(np.abs((actual_mw-predicted_mw)/actual_mw))*100),
    'R2': float(1-np.sum((actual_mw-predicted_mw)**2)/np.sum((actual_mw-actual_mw.mean())**2)),
    'test_samples': int(len(p)),
}
print(json.dumps(metrics, indent=2))
predictions = pd.DataFrame({'timestamp': birmingham.timestamp.iloc[p].to_numpy(), 'actual_birmingham_mw': actual_mw, 'birmingham_baseline_mw': np.exp(birmingham.log_baseline.to_numpy()[p]), 'predicted_residual_log': predicted_residual, 'predicted_birmingham_mw': predicted_mw})
predictions.to_csv(transfer_dir / 'birmingham_2023_cross_city_predictions.csv', index=False)
(transfer_dir / 'birmingham_2023_cross_city_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
assert pd.Timestamp(predictions.timestamp.min()) > VALIDATION_END
print('Leakage check passed: every evaluated Birmingham target is in 2023.')


137/137 [==============================] - 2s 8ms/step
{
  "experiment": "Atlanta weather-residual TCN to Birmingham 2023",
  "MAE_MW": 12.595627591259596,
  "RMSE_MW": 17.176520480976457,
  "MAPE_percent": 1.0116967075926495,
  "R2": 0.9954789509107697,
  "test_samples": 8760
}
Leakage check passed: every evaluated Birmingham target is in 2023.


## Saved artifacts

The results folder contains the independent Atlanta and Birmingham baseline models and scalers, the Atlanta-trained TCN, source-fitted TCN scalers, training history, configuration, Birmingham 2023 predictions, and final metrics. For the reverse direction, change the two city configuration values, restart the kernel, and run every cell.
